In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadMerge\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadtest\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Platform.Utils.Geom;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XdgTimestepping;
Init();

In [2]:
using BoSSS.Foundation.Grid.Classic;
using ilPSP.Utils;
using System;
using System.Collections.Generic;
using System.Linq;
using System.Text;
using System.Threading.Tasks;
using BoSSS.Foundation.Grid;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.Control;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
using BoSSS.Solution.XNSECommon;

In [3]:
//ExecutionQueues

In [4]:
//GetDefaultQueue()

In [5]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("EE-Cylinder-Beam");

In [6]:
BoSSSshell.WorkflowMgm.Init("EE-Cylinder-Beam");

Project name is set to 'EE-Cylinder-Beam'.
Default Execution queue is chosen for the database.
Opening existing database 'C:\Users\miao\Documents\Database\EE-Cylinder-Beam'.


In [7]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

In [8]:
//wmg.DefaultDatabase

## Create grid

In [9]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xLeft = -0.3;
        double xRight = 2.5;
        double ySize = 0.4;
        int kelem = 20;

        double[] CutOut1Point1 = new double[2] {  0.15, 0.25 }; 
        double[] CutOut1Point2 = new double[2] {  0.25, 0.15 };
        
        var CutOut1 = new BoSSS.Platform.Utils.Geom.BoundingBox(2);
        CutOut1.AddPoint(CutOut1Point1);
        CutOut1.AddPoint(CutOut1Point2);

        double[] Xnodes = GenericBlas.Linspace(xLeft, xRight, (int)((xRight - xLeft) * kelem) + 1);
        double[] Ynodes = GenericBlas.Linspace(0, ySize, (int)(ySize * kelem) + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, CutOuts: CutOut1);

        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "wall-upper");
        grd.EdgeTagNames.Add(3, "velocity_inlet_left");
        grd.EdgeTagNames.Add(4, "pressure_outlet_right");
        grd.EdgeTagNames.Add(5, "wall_inside");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 5;

            if(Math.Abs(X[1] - 0) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - ySize) <= 1.0e-8)
                et = 2;
            if(Math.Abs(X[0] - xLeft) <= 1.0e-8)
                et = 3;
            if(Math.Abs(X[0] - xRight) <= 1.0e-8)
                et = 4;

            return et;
        });

        return grd;
     }
 
 }

In [10]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double InletVelocity(double[] X) {");
           stw.WriteLine("    return 1.5 * 1.0 * X[1] * (0.4 - X[1]) / (0.4 / 2).Pow2();");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           //stw.WriteLine("    return ((X[0] - 0.2).Pow2() + (X[1] - 0.2).Pow2()) - 0.05 * 0.05;");
           stw.WriteLine("    return -1.0;");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi2(double[] X) { ");
           stw.WriteLine("    double h = 0.01;");
           stw.WriteLine("    if(X[1] < 0.2 && X[0] >= (0.25) && X[0] < (0.6 - h)) {");
           stw.WriteLine("        return -(X[1] - 0.2).Pow2() + h * h;");
           stw.WriteLine("    }");
           stw.WriteLine("    if(X[1] > 0.2 && X[0] >= (0.25) && X[0] < (0.6 - h)) {");
           stw.WriteLine("        return -(X[1] - 0.2).Pow2() + h * h;");
           stw.WriteLine("    }");
           stw.WriteLine("    return -((X[1] - 0.2).Pow2() + (X[0] - (0.6 - h)).Pow2()) + h * h;");
           //stw.WriteLine("    return -1.0;");
           stw.WriteLine("    }"); 

           stw.WriteLine(" static public double InitialVelocityVx(double[] X) { ");
           stw.WriteLine("    return 1.5 * 1.0 * X[1] * (0.4 - X[1]) / (0.4 / 2).Pow2() - 10 * Convert.ToInt32(((X[0]+0.1)*(X[0]+0.1)+(X[1]-0.2)*(X[1]-0.2))<=0.01) * (X[1]-0.2);");
           stw.WriteLine("    }"); 
            
           stw.WriteLine(" static public double InitialVelocityVy(double[] X) { ");
           stw.WriteLine("    return 0.0 + 10 * Convert.ToInt32(((X[0]+0.1)*(X[0]+0.1)+(X[1]-0.2)*(X[1]-0.2))<=0.01) * (X[0]+0.1);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InletVelocity(){
        return new Formula("BoundaryValues.InletVelocity", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi2(){
        return new Formula("BoundaryValues.InitialPhi2", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialVelocityVx(){
        return new Formula("BoundaryValues.InitialVelocityVx", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialVelocityVy(){
        return new Formula("BoundaryValues.InitialVelocityVy", AdditionalPrefixCode:GetPrefixCode());
    }
    
}

## Create Control file

In [11]:
public static ZLS_Control Channel( int p = 2, int AMRlvl = 2, double BeamDensity = 1) {
    ZLS_Control C = new ZLS_Control(p);
    C.AgglomerationThreshold = 0.2;
    C.NoOfMultigridLevels = 1;
    int D = 2;
    //AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

    #region db
    C.savetodb = true;
    C.ProjectName = "Cylinder-Beam";
    C.SessionName = "Cylinder-Beam-Comsol-Cluster";
    //C.ProjectDescription = "Droplet running on pc";
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());
    #endregion
    
    // Physical Parameters
    // ===================
    #region physics
    C.PhysicalParameters.rho_A = 1e1;
    //C.PhysicalParameters.rho_B = 1e1;
    C.PhysicalParameters.mu_A = 1e-2;
    //C.PhysicalParameters.mu_B = 1e-3;
    //double sigma = 0.046;
    //C.PhysicalParameters.Sigma = sigma;
    //C.PhysicalParameters.betaS_A = 0.0;
    //C.PhysicalParameters.betaS_B = 0.0;
    //C.PhysicalParameters.betaL = 0.0;
    //C.PhysicalParameters.theta_e = Math.PI / 2.0;
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false; //??
    C.Material = new Solid() {
        Density = BeamDensity,
        //Lame2 = 0.5e6,
        Lame2 = 1e3,
        PoissonsRatio = 0.5,
        Viscosity = 1e-2
    };
    #endregion
    // grid generation
    // ===============
    #region grid
    C.SetGrid(GridFactory.GenerateGrid());
    #endregion
    // Initial Values
    // ==============
    //
    C.AddInitialValue("VelocityX#A", BoundaryValueFactory.Get_InitialVelocityVx());
    //C.AddInitialValue("VelocityX#B", BoundaryValueFactory.Get_InitialVelocityVx());
    C.AddInitialValue("VelocityY#A", BoundaryValueFactory.Get_InitialVelocityVy());
    //C.AddInitialValue("VelocityY#B", BoundaryValueFactory.Get_InitialVelocityVy());
    C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
    C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi2());
    //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi2());
    // boundary conditions
    // ===================
    #region BC
    C.AddBoundaryValue("wall_lower", "VelocityX#A", BoundaryValueFactory.Get_Zero());
    //C.AddBoundaryValue("wall_lower", "VelocityX#B", BoundaryValueFactory.Get_Zero());    
    C.AddBoundaryValue("wall-upper", "VelocityX#A", BoundaryValueFactory.Get_Zero());
    //C.AddBoundaryValue("wall-upper", "VelocityX#B", BoundaryValueFactory.Get_Zero());
    C.AddBoundaryValue("velocity_inlet_left", "VelocityX#A", BoundaryValueFactory.Get_InletVelocity());
    //C.AddBoundaryValue("velocity_inlet_left", "VelocityX#B", BoundaryValueFactory.Get_InletVelocity());
    C.AddBoundaryValue("pressure_outlet_right");
    C.AddBoundaryValue("wall_inside");
    C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    //C.PhysicalParameters.sliplength = 0.001;
    #endregion
    // misc. solver options
    // ====================
    #region solver
    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;
    C.NonLinearSolver.MaxSolverIterations = 80;
    C.NonLinearSolver.MinSolverIterations = 2;
    //C.Solver_MaxIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-12;
    //C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
    //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
    //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
    //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
    //C.fullReInit = false; 
    C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
    C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.Standard;
    
    C.AdaptiveMeshRefinement = true;
    double[] p11 = new double[2] {0.1, 0.1};
    double[] p12 = new double[2] {0.3, 0.3};   
    var ind1     = new BoSSS.Application.XNSE.AMRInBoundingBox(p11, p12);
    ind1.maxRefinementLevel = AMRlvl - 1;
    C.activeAMRlevelIndicators.Add(ind1);
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl });
    C.AMR_startUpSweeps = AMRlvl;

    //double[] p11 = new double[2] {0.0, 0.0};
    //double[] p12 = new double[2] {0.4, 0.4};   
    //var refineMesh = new BoundingBox(p11, p12);
    //var ind1     = new BoSSS.Solution.AMRLevelIndicatorLibrary.AMRInBoundingBox(refineMesh);
    //ind1.maxRefinementLevel = AMRlvl;
    //C.activeAMRlevelIndicators.Add(ind1);

    //C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRInBoundingBox(new BoundingBox(new double[,] { { 0.4, 0.4 }, { 0, 0 } })) { maxRefinementLevel = AMRlvl });

    #endregion

    C.DynamicLoadBalancing_On = false;
    C.DynamicLoadBalancing_Period = 50;
    C.DynamicLoadBalancing_RedistributeAtStartup = false;

    //C.GridPartType = GridPartType.clusterHilbert;
    C.GridPartType = GridPartType.METIS;

    // Timestepping
    // ============
    #region time
    //C.CheckJumpConditions = true;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    //C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;

    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
    
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    double dt = 1e-2;
    C.dtMax = dt;
    C.dtMin = dt;
    C.NoOfTimesteps = 1000;
    C.saveperiod = 1;
    #endregion
    return C;
}

## Send and run jobs

In [12]:
//double[] Density = new double[] {0.1, 1, 10, 100}; 
double[] Density = new double[] {1e1}; 

In [13]:
foreach(double Den in Density){
    var C_CTRLFILE = Channel(2, 2, Den);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = 4;
    JobLocal.Activate();
    JobLocal.ShowOutput(); 
}



Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Cylinder-Beam-Comsol-Cluster ... 
Opening existing database '\\dc3\userspace\miao\cluster\EE-Cylinder-Beam'.
Set Database: { Session Count = 23; Grid Count = 636; Path = C:\Users\miao\Documents\Database\EE-Cylinder-Beam }
Grid is not in database yet...
Grid successfully saved: 3e3a9774-57d4-48cc-83b7-4ea4cb8b9dfd


Deploying executables and additional files ...
Deployment directory: C:\Users\miao\Documents\Database\EE-Cylinder-Beam-ZwoLevelSetSolver2024Mar15_153001.862022
copied 46 files.
   written file: control.obj
deployment finished.
Mini batch processor is already running.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [14]:
wmg.Sessions

#0: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 3:30:21 PM	bfc042ef...
#1: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 3:23:30 PM	66d189fb...
#2: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 3:21:00 PM	38c78331...
#3: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 3:13:07 PM	7dedea6b...
#4: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 3:03:05 PM	fcbc0ee7...
#5: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 1:41:44 PM	f6487264...
#6: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 1:36:01 PM	88e34401...
#7: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 1:34:28 PM	72ed57f5...
#8: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 1:30:56 PM	73f0a9b6...
#9: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 1:25:33 PM	dbfcffdb...
#10: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 1:19:31 PM	9dbefc42...
#11: EE-Cylinder-Beam	Cylinder-Beam-Comsol-Cluster*	3/15/2024 1:15:32 PM	e0

In [15]:
wmg.Sessions[0].Timesteps.Count

9

In [16]:
var outPath = wmg.Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\EE-Cylinder-Beam__Cylinder-Beam-Comsol-Cluster__bfc042ef-5586-40b9-925a-592638c3cf40


## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [ ]:
databases[0].Sessions

In [ ]:
var session = databases[0].Sessions[0];

In [ ]:
session.Timesteps.Count

In [ ]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("VelocityY").ProbeAt(0.405, 0),
        t => "VelocityY"
        );

In [ ]:
mydataset.PlotNow()

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(518).Export().WithSupersampling(3).Do();

In [ ]:
var DispY = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementY").Single();

In [ ]:
DispY

In [ ]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("Phi2"),
        t => "DisplacementY"
        );

In [ ]:
mydataset.PlotNow()

## Restart

In [ ]:
databases[0].Sessions

In [ ]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = databases[0].Sessions[0];
Sloc

In [ ]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [ ]:
c2.GetType()

In [ ]:
c2.GridGuid

In [ ]:
Sloc.Timesteps.Last().GridID

In [ ]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [ ]:
c2.GridGuid

In [ ]:
//c2.DynamicLoadBalancing_On = true;
//c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
//c2.GridPartType = GridPartType.METIS;

//c2.AdaptiveMeshRefinement = true;
//c2.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
//c2.AMR_startUpSweeps = 5;
c2.AdaptiveMeshRefinement = false;

In [ ]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [ ]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

In [ ]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

In [ ]:
databases[0].Sessions

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

In [ ]:
databases[0].Sessions[0].